In [45]:
import pyEDM
import pandas as pd
import numpy as np
import random

p22 = pd.read_csv('../data/processed_22.csv')
p23 = pd.read_csv('../data/processed_23.csv')
p24 = pd.read_csv('../data/processed_24.csv')
p25 = pd.read_csv('../data/processed_25.csv')
combined = pd.concat([p22, p23, p24, p25], ignore_index=True)

In [46]:
env = pd.read_csv('../data/environment_all.csv')
env = env.drop(columns=['fluorescent_dissolved_organic_matter_eco', 'sea_water_turbidity_eco', 'waterlevel_predicted_m'])

# ones of contention (due to too many missing values):
    # mass_concentration_of_oxygen_in_sea_water_seaphox
    # mole_concentration_of_dissolved_molecular_oxygen_in_sea_water_seaphox
    # fractional_saturation_of_oxygen_in_sea_water_seaphox
    # sea_water_ph_reported_on_total_scale_seaphox_external

env = env.drop(columns=['mass_concentration_of_oxygen_in_sea_water_seaphox', 'mole_concentration_of_dissolved_molecular_oxygen_in_sea_water_seaphox', 'fractional_saturation_of_oxygen_in_sea_water_seaphox', 'sea_water_ph_reported_on_total_scale_seaphox_external'])
env.head()

,date,mass_concentration_of_chlorophyll_in_sea_water_ctd,sea_water_electrical_conductivity_ctd,sea_water_practical_salinity_ctd,sea_water_sigma_t_ctd,sea_water_pressure_ctd,sea_water_temperature_ctd,wind_speed_ms,wind_dir_deg,wind_gust_ms,air_temp_c,baro_mb,waterlevel_verified_m
0,20221001,28.717,47.012,33.406,23.250,3.475,21.054,2.066667,216.458333,3.166667,19.250000,1010.879167,0.955167
1,20221002,37.439,46.926,33.418,23.294,3.454,20.957,2.287500,217.166667,3.050000,19.841667,1012.595833,0.927333
2,20221003,46.564,45.266,33.377,23.694,3.456,19.307,2.279167,235.750000,3.087500,20.229167,1014.079167,0.899667
3,20221004,55.428,43.889,33.317,23.990,3.439,17.962,1.491667,172.500000,2.050000,19.504167,1013.445833,0.903083
4,20221005,62.660,44.128,33.304,23.917,3.427,18.227,1.645833,238.833333,2.045833,18.700000,1013.991667,0.892917


In [61]:
features = [
    "Lpoly_expected_ml", "Area", "Biovolume", "MajorAxisLength",
    "MinorAxisLength", "Perimeter", "Orientation", "Eccentricity",
    "Solidity", "texture_uniformity", "texture_smoothness",
    "texture_average_gray_level", "texture_entropy",
    "texture_average_contrast", "H90", "H180", "Hflip",
    "Extent", "EquivDiameter", "ConvexArea", "ConvexPerimeter"
]

df = combined[["date"] + features].copy()
df["date"] = pd.to_datetime(df["date"].astype(str), format="%Y%m%d")
df = df.sort_values("date").set_index("date")

# Force shared daily index
df = df.asfreq("D")

df_filled = df.copy()
was_imputed = pd.DataFrame(index=df.index)

for col in features:
    ema = df[col].ewm(span=30, adjust=False).mean()
    df_filled[col] = df[col].fillna(ema)
    was_imputed[col] = df[col].isna()

# normalize features
df_norm = df_filled.copy()
for col in features:
    mu = df_filled[col].mean()
    sigma = df_filled[col].std()
    
    df_norm[col] = (df_filled[col] - mu) / sigma

df_mv = df_norm.copy()
df_mv = df_mv.reset_index()
df_mv["t"] = np.arange(1, len(df_mv) + 1)
df_mv = df_mv[["t"] + features]

target = "Lpoly_expected_ml"
predictors = [col for col in features if col != target]

https://sugiharalab.github.io/EDM_Documentation/parameters/

In [89]:
def one_simplex(df, target, features, E=4, Tp=1):
    # Randomly select 3 features (+ the target) for the simplex projection
    chosen_features = random.sample(features, 3)
    columns = [target] + chosen_features
    columns_str = " ".join(columns) # has to be 'space separated' idk ????

    N = len(df)
    res = pyEDM.Simplex(
        dataFrame=df,
        columns=columns_str,
        target=target,
        E=E,
        tau=1,
        Tp=Tp,
        lib=f"1 {N}",
        pred=f"1 {N}"
    )

    obs = res["Observations"].to_numpy()
    pred = res["Predictions"].to_numpy()

    mask = np.isfinite(obs) & np.isfinite(pred)
    obs = obs[mask]
    pred = pred[mask]

    if len(obs) < 10 or np.std(obs) == 0 or np.std(pred) == 0:
        return np.nan, chosen_features

    rho = np.corrcoef(obs, pred)[0, 1]
    rmse = np.sqrt(np.mean((obs - pred) ** 2))
    mae = np.mean(np.abs(obs - pred))
    return rho, rmse, mae, chosen_features

def multiview_big(df, target, features, Tp, n_trials=500):
    results = []

    for i in range(n_trials):
        rho, rmse, mae, chosen = one_simplex(df, target, features, E=4, Tp=Tp)

        results.append({
            "rho": rho,
            "rmse": rmse,
            "mae": mae,
            "features": chosen
        })

    return pd.DataFrame(results)

In [90]:
from collections import Counter

x = df_mv[target].to_numpy()

summary_rows = []
feature_importance_by_tp = {}

for Tp in range(1, 32):
    mv = multiview_big(df_mv, target, predictors, Tp, n_trials=500)

    acf = abs(pd.Series(x).autocorr(lag=Tp))
    mv["rho_eff"] = mv["rho"] - acf

    # summary stats
    summary_rows.append({
        "Tp": Tp,
        "rho_eff_mean": mv["rho_eff"].mean(),
        "rmse_mean": mv["rmse"].mean(),
        "mae_mean": mv["mae"].mean(),
        "n_models": len(mv)
    })

    # feature importance (top 5%)
    top = mv.nlargest(int(0.05 * len(mv)), "rho_eff")

    counts = Counter()
    for feats in top["features"]:
        for f in feats:
            counts[f] += 1

    importance = pd.Series(counts) / len(top)
    feature_importance_by_tp[Tp] = importance.sort_values(ascending=False)

In [91]:
summary_df = pd.DataFrame(summary_rows)
summary_df

,Tp,rho_eff_mean,rmse_mean,mae_mean,n_models
0,1,0.189587,0.440028,0.083862,500
1,2,0.441783,0.403211,0.078743,500
2,3,0.584699,0.377971,0.080460,500
3,4,0.270783,0.810417,0.181902,500
4,5,0.125533,0.945518,0.216849,500
5,6,0.165825,0.965110,0.226251,500
6,7,0.147915,0.979086,0.236489,500
7,8,0.081031,0.998370,0.240146,500
8,9,0.102518,0.996812,0.239738,500
9,10,0.077253,0.997229,0.243745,500


In [92]:
importance_all = pd.concat(
    feature_importance_by_tp,
    names=["Tp", "Feature"]
).reset_index()

importance_all.columns = ["Tp", "Feature", "Proportion"]
importance_all

,Tp,Feature,Proportion
0,1,Solidity,0.52
1,1,Orientation,0.48
2,1,Area,0.32
3,1,texture_average_gray_level,0.24
4,1,MajorAxisLength,0.24
...,...,...,...
473,31,texture_average_gray_level,0.12
474,31,EquivDiameter,0.12
475,31,Biovolume,0.08
476,31,Eccentricity,0.08
